# Part 1: Analyzing Best-Seller Trends Across Product Categories
Objective: Understand the relationship between product categories and their best-seller status.

1. Crosstab Analysis:

- Create a crosstab between the product category and the isBestSeller status.

- Are there categories where being a best-seller is more prevalent?

Hint: one option is to calculate the proportion of best-sellers for each category and then sort the categories based on this proportion in descending order.

In [ ]:
import pandas as pd
import plotly.express as px

In [ ]:
df = pd.read_csv("amz_uk_price_prediction_dataset.csv")

In [ ]:
df.head(10)

## Crosstab between product category and bestseller status

In [ ]:
pd.crosstab(df["category"],df["isBestSeller"])


In [ ]:
ct = pd.crosstab(df["category"], df["isBestSeller"])
ct_sorted = ct.sort_values(by=True, ascending=False)
ct_sorted


In [ ]:
ct_prop = pd.crosstab(
    df["category"],
    df["isBestSeller"],
    normalize="index"
)

ct_prop.sort_values(by=True, ascending=False).head()


## Findings

- A best-seller is the item with the highest sales volume in specific categories.
- Best-seller is more prevalent in the above 5 categories compared to the others in the dataset.
- Grocery falls under basic needs and hence this could drive the sales volume.
- Mobile phone accessories, Smart home security and lighting in the top % list denotes the prevalence of use of technology that has widened in the recent days, especially with Alexa as an example from Amazon.
- The products that are (highly) rated could also be one of the factors driving the sales volume and people tend to leave reviews for things they consume (like food and groceries), electronics (working precision, etc.,) more often.


2. Statistical Tests:

- Conduct a Chi-square test to determine if the best-seller distribution is independent of the product category.
- Compute Cramér's V to understand the strength of association between best-seller status and category.

## Statistical Tests

### Chi-square test - to check if best-seller distrbituion is independent of product category

-  H0: Best-seller status is independent of product category

- Ha: Best-seller status depends on product category

In [ ]:
from scipy.stats import chi2_contingency

alpha = 0.05

chi2, p_value, dof, expected = chi2_contingency(ct)


print(f"Chi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.6f}")



- The p-value is less than the significance value, hence null hypothesis is rejected. The best seller status depends on the product category.

### Cramér's V - to understand the association between best-seller status and product category

In [ ]:
n = ct.sum().sum()
r, k = ct.shape

cramers_v = np.sqrt(chi2 / (n * (min(r, k) - 1)))
print(f"Cramér's V: {cramers_v:.4f}")


- Since the Cramér's V value is 0.1223, considering the thumb rule that any value between 0.10 – 0.30 means a weak association, it is safe to say that the association between the best seller status and the product category is weak.

3. Visualizations:

- Visualize the relationship between product categories and the best-seller status using a stacked bar chart.

In [ ]:
ct = pd.crosstab(df["category"], df["isBestSeller"])
ct["total"] = ct[False] + ct[True]

ct_top10 = ct.sort_values(by="total", ascending=False).head(10).reset_index()

fig = px.bar(
    ct_top10,
    x="category",
    y=[False, True],
    barmode="stack",
    title="Best-Seller Status by Top 10 Product Categories",
    labels={
        "value": "Number of Products",
        "variable": "Best-Seller Status",
        "category": "Product Category"
    }
)
fig.show()

# Part 2: Exploring Product Prices and Ratings Across Categories and Brands
Objective: Investigate how different product categories influence product prices.

0. Preliminary Step: Remove outliers in product prices.

For this purpose, we can use the IQR (Interquartile Range) method. Products priced below the first quartile minus 1.5 times the IQR or above the third quartile plus 1.5 times the IQR will be considered outliers and removed from the dataset. The next steps will be done with the dataframe without outliers.

Hint: you can check the last Check For Understanding at the end of the lesson EDA Bivariate Analysis for a hint on how to do this.

In [ ]:
#calculating IQR
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)
IQR = Q3 - Q1

Q1, Q3, IQR

In [ ]:
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

lower_bound, upper_bound


In [ ]:
#eliminating outliers using the upper and lowerbound
df_no_outliers = df[
    (df["price"] >= lower_bound) &
    (df["price"] <= upper_bound)
]


1. Violin Plots:

- Use a violin plot to visualize the distribution of price across different product categories. Filter out the top 20 categories based on count for better visualization.

In [ ]:
top_20_categories = (
    df_no_outliers["category"]
    .value_counts()
    .head(20)
    .index
)
df_top20 = df_no_outliers[
    df_no_outliers["category"].isin(top_20_categories)
]

fig = px.violin(
    df_top20,
    x="category",
    y="price",
    box=True, #shows median line, IQR and skewness
    points=False,
    title = "Price Distribution across top 20 Categories (without outliers)"
)

fig.show()

## Findings

- The category 'men', which would be from the wider categorisation (men, women, children) has the highest median of 20.99.
- This denotes that cheaper items in this category is not common and the price is typically higher.

- Which product category tends to have the highest median price? Don't filter here by top categories.

In [ ]:
median_by_category = (
    df_no_outliers
    .groupby("category")["price"]
    .median()
    .sort_values(ascending=False)
)

median_by_category.head()

In [ ]:
fig = px.violin(
    df_no_outliers,
    x="category",
    y="price",
    box=True,
    points=False,
    title="Price Distribution by Product Category (without outliers)"
)

fig.update_layout(xaxis_tickangle=45)
fig.show()


- Desktop PCs, Tablets, Motherboards, Graphic Cards are all related to electronic devices or accessories.
- So electronic wares and accessories tend to have a higher median 


2. Bar Charts:

- Create a bar chart comparing the average price of products for the top 10 product categories (based on count).

In [ ]:
top_10_categories = (
    df["category"]
    .value_counts()
    .head(10)
    .index
) 

In [ ]:
df_top10 = df[df["category"].isin(top_10_categories)]

In [ ]:
avg_price_by_category = (
    df_top10
    .groupby("category")["price"]
    .mean()
    .reset_index()
)

In [ ]:
avg_price_by_category = avg_price_by_category.sort_values(by="price", ascending=False)
fig = px.bar(
    avg_price_by_category,
    x="category",
    y="price",
    title="Average Product Price for Top 10 Categories",
    labels={
        "category": "Product Category",
        "price": "Average Price (£)"
    }
)

fig.update_layout(xaxis_tickangle=45)
fig.show()

- Which product category commands the highest average price? Don't filter here by top categories.

In [ ]:
avg_price_by_category = (
    df
    .groupby("category")["price"]
    .mean()
    .sort_values(ascending=False)
)

avg_price_by_category.head()


- Based on the average price calculated across all product categories, the category with the highest average price is Laptops, indicating that products in this category are generally more expensive than others.
- The previous analysis of the highest median price per category also belonged to electronic devices and accessories. This reiterates the former.

3. Box Plots:

- Visualize the distribution of product ratings based on their category using side-by-side box plots. Filter out the top 10 categories based on count for better visualization.
- Which category tends to receive the highest median rating from customers? Don't filter here by top categories.

In [ ]:
top_10_categories = (
    df["category"]
    .value_counts()
    .head(10)
    .index
)
df_top10 = df[df["category"].isin(top_10_categories)]

fig = px.box(
    df_top10,
    x="category",
    y="stars",
    title="Distribution of Product Ratings by Top 10 Categories",
    labels={
        "category": "Product Category",
        "rating": "Product Rating"
    }
)

fig.update_layout(xaxis_tickangle=45)
fig.show()

- Skin-care, Bath and body, Beauty, Hair-care have a median rating of 4.4. This high median shows that there are more hihgly rated products in these categories.

In [ ]:
median_rating_by_category = (
    df
    .groupby("category")["stars"]
    .median()
    .sort_values(ascending=False)
)
median_rating_by_category.head()

# Part 3: Investigating the Interplay Between Product Prices and Ratings
Objective: Analyze how product ratings (stars) correlate with product prices.

Correlation Coefficients:

- Calculate the correlation coefficient between price and stars.
- Is there a significant correlation between product price and its rating?

## Correlation between price and stars

In [ ]:
# Drop missing values
df_corr = df[['price', 'stars']].dropna()


### Pearson correlation

In [ ]:
pearson_corr = df_corr['price'].corr(df_corr['stars'], method='pearson')
print(f"Pearson correlation: {pearson_corr:.4f}")

### Spearman correlation

In [ ]:
spearman_corr = df_corr['price'].corr(df_corr['stars'], method='spearman')
print(f"Spearman correlation: {spearman_corr:.4f}")

- Both the correlation coefficients are negative, so there is a negative correlation between price and stars. When price increases, star / rating decreases, or in this context, higher price products are associated with lower ratings.
- The relationship is roughly monotonic and linear, weak but consistent. 

## Significance of Correlation between price and reviews

- H0: The correlation between price and reviews is not statistically significant
- Ha: The correlation between price and reviews is statistically significant

In [ ]:
from scipy.stats import pearsonr, spearmanr

alpha = 0.05

pearson_r, pearson_p = pearsonr(df_corr['price'], df_corr['stars'])
spearman_r, spearman_p = spearmanr(df_corr['price'], df_corr['stars'])

print(f"Pearson r = {pearson_r:.4f}, p-value = {pearson_p:.6f}")
print(f"Spearman r = {spearman_r:.4f}, p-value = {spearman_p:.6f}")


- Since the p-value is smaller than the significance value, we reject the null hypothesis. So, the correlation between price and reviews is statistically significant

Visualizations:

- Use a scatter plot to visualize the relationship between product rating and price. What patterns can you observe?
- Use a correlation heatmap to visualize correlations between all numerical variables.
- Examine if product prices typically follow a normal distribution using a QQ plot.


## Scatter Plot

In [ ]:
fig = px.scatter(
    df,
    x="stars",
    y="price",
    title="Relationship Between Product Rating and Price",
    labels={
        "stars": "Product Rating",
        "price": "Product Price (£)",
        
    },
    opacity=0.4
)

fig.show()

## Findings

- There is no clear linear relationship between product rating and price.
- High ratings appear at both lower and higher price points.
- So price does not strongly influence product ratings.
- The plottings at 0 could be products with no ratings / missing ratings.

## Heat Map

In [ ]:
numerical_df = df.select_dtypes(include="number")
corr_matrix = numerical_df.corr()
fig = px.imshow(
    corr_matrix,
    text_auto=True,
    title="Correlation Heatmap of Numerical Variables",
    labels=dict(color="Correlation")
)

fig.show()


- Most correlation values are closer to 0, indicating a lack of strong relationships between the numerical variables

## QQ plot

In [ ]:
!pip install statsmodels

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

price = df['price'].dropna()

# sampling data since the population is huge
sample = price.sample(n=10_000, random_state=42)

sm.qqplot(sample, line='s')
plt.title("QQ Plot of Product Prices")
plt.show()



- Product prices do not follow a normal deviation. There is a sharp upward increase and the heavy right tail indicates outliers.